# Dependencies

In [3]:
!pip install paho-mqtt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.7 MB/s eta 0:00:00


# Calibrating script

In [8]:
import json
import time
import paho.mqtt.client as mqtt
import threading
import sys

id = '<ID>'

client_telemetry_topic = id + '/telemetry'
server_command_topic = id + '/commands'
client_name = id + 'calibration_server'

# Fix for newer paho-mqtt version
try:
    # Try version 2.0+ syntax
    mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_name)
except AttributeError:
    # Fall back to older syntax
    mqtt_client = mqtt.Client(client_name)

mqtt_client.connect('test.mosquitto.org')
mqtt_client.loop_start()

# Calibration settings
water_time = 1  # 1 second per test
wait_time = 15  # Wait for moisture to stabilize
max_tests = 6   # How many 1-second increments to test

# Store results
results = []
current_test = 0
waiting_for_stabilization = False
test_start_time = 0
calibration_active = False

def send_relay_command(client, state):
    """Send relay command to turn pump on/off"""
    command = {'relay_on': state}
    print(f"Sending: {command}")
    client.publish(server_command_topic, json.dumps(command))

def start_calibration():
    global current_test, waiting_for_stabilization, calibration_active
    print("\n=== CALIBRATION STARTED ===")
    print("Starting with dry soil...")
    print(f"Will add {water_time} second(s) of water, {max_tests} times")
    print("=" * 40)
    current_test = 0
    waiting_for_stabilization = False
    calibration_active = True

def record_reading(soil_moisture):
    global current_test, waiting_for_stabilization, test_start_time, calibration_active

    if not calibration_active:
        return

    if current_test == 0:
        # Initial dry reading
        results.append({
            'total_pump_time': 'Dry',
            'soil_moisture': soil_moisture,
            'decrease': 0
        })
        print(f"\n[DRY BASELINE] Soil moisture: {soil_moisture}")
        print("Starting first watering cycle...")
        send_relay_command(mqtt_client, True)
        time.sleep(water_time)
        send_relay_command(mqtt_client, False)
        waiting_for_stabilization = True
        test_start_time = time.time()
        current_test = 1
    else:
        # After watering, wait for stabilization
        if waiting_for_stabilization:
            elapsed = time.time() - test_start_time
            if elapsed >= wait_time:
                # Take reading
                previous_moisture = results[-1]['soil_moisture']
                decrease = previous_moisture - soil_moisture

                results.append({
                    'total_pump_time': f"{current_test}s",
                    'soil_moisture': soil_moisture,
                    'decrease': decrease
                })

                print(f"\n[TEST {current_test}] After {current_test}s total water:")
                print(f"  Moisture: {soil_moisture}")
                print(f"  Decrease: {decrease}")

                # Check if we're done
                if current_test >= max_tests:
                    print_calibration_results()
                    calibration_active = False
                    return

                # Start next watering cycle
                print(f"\nStarting next watering cycle...")
                send_relay_command(mqtt_client, True)
                time.sleep(water_time)
                send_relay_command(mqtt_client, False)
                waiting_for_stabilization = True
                test_start_time = time.time()
                current_test += 1
            else:
                # Show progress
                remaining = wait_time - elapsed
                if remaining > 0:
                    print(f"  Stabilizing... waiting {remaining:.0f}s more", end='\r')

def print_calibration_results():
    print("\n\n" + "="*50)
    print("CALIBRATION COMPLETE")
    print("="*50)
    print(f"\n{'Total Pump time':<20} {'Soil Moisture':<15} {'Decrease':<10}")
    print("-"*50)

    decreases = []
    for r in results:
        if r['total_pump_time'] != 'Dry':
            decreases.append(r['decrease'])
        print(f"{r['total_pump_time']:<20} {r['soil_moisture']:<15} {r['decrease']:<10}")

    if decreases:
        avg_decrease = sum(decreases) / len(decreases)
        print("-"*50)
        print(f"\nAverage decrease per second: {avg_decrease:.2f}")
        print(f"\nUpdate your server code with:")
        print(f"   AVG_DECREASE_PER_SEC = {avg_decrease:.2f}")
        print("="*50)

        # Save to file for later use
        calibration_data = {
            'avg_decrease_per_sec': avg_decrease,
            'results': results,
            'calibration_date': time.strftime("%Y-%m-%d %H:%M:%S"),
            'sensor_type': 'Capacitive Soil Moisture Sensor',
            'water_time_per_test': water_time,
            'stabilization_time': wait_time
        }

        with open('calibration_data.json', 'w') as f:
            json.dump(calibration_data, f, indent=2)
        print("\nCalibration saved to calibration_data.json")
    else:
        print("\nNo decrease data collected!")

def handle_telemetry(client, userdata, message):
    """Handle incoming telemetry messages"""
    try:
        payload = json.loads(message.payload.decode())
        soil_moisture = payload.get('soil_moisture')

        if soil_moisture is not None:
            record_reading(soil_moisture)
    except Exception as e:
        print(f"Error handling telemetry: {e}")

# Setup MQTT message handling
mqtt_client.subscribe(client_telemetry_topic)
mqtt_client.on_message = handle_telemetry

print("="*50)
print("SOIL MOISTURE CALIBRATION SERVER")
print("="*50)
print(f"Telemetry topic: {client_telemetry_topic}")
print(f"Command topic: {server_command_topic}")
print("\nIMPORTANT:")
print("1. Make sure your soil is DRY before starting")
print("2. Ensure your IoT device is publishing telemetry")
print("3. The pump should be connected to the relay")
print("="*50)

# Wait for user to start
input("\nPress Enter to start calibration...")
start_calibration()

try:
    while calibration_active:
        time.sleep(1)
    print("\nCalibration finished! You can now stop this script.")
    print("Run your improved server script with the calibration data.")
except KeyboardInterrupt:
    print("\n\nCalibration stopped by user")
    if len(results) > 1:
        print_calibration_results()
    else:
        print("No calibration data collected")
finally:
    mqtt_client.loop_stop()
    mqtt_client.disconnect()

SOIL MOISTURE CALIBRATION SERVER
Telemetry topic: <ID>/telemetry
Command topic: <ID>/commands

IMPORTANT:
1. Make sure your soil is DRY before starting
2. Ensure your IoT device is publishing telemetry
3. The pump should be connected to the relay

Press Enter to start calibration...

=== CALIBRATION STARTED ===
Starting with dry soil...
Will add 1 second(s) of water, 6 times


Calibration stopped by user
No calibration data collected


# Server

In [9]:
import json
import time
import paho.mqtt.client as mqtt
import threading
import os

id = '<ID>'

client_telemetry_topic = id + '/telemetry'
server_command_topic = id + '/commands'
client_name = id + 'soilmoistureserver'

# Fix for newer paho-mqtt version
try:
    # Try version 2.0+ syntax
    mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_name)
except AttributeError:
    # Fall back to older syntax
    mqtt_client = mqtt.Client(client_name)

mqtt_client.connect('test.mosquitto.org')
mqtt_client.loop_start()

# ===== CALIBRATION CONSTANTS =====
AVG_DECREASE_PER_SEC = 18.83  # Default from calibration data
TARGET_MOISTURE = 450         # Target soil moisture level
SAFETY_FACTOR = 0.9          # Run 90% of calculated time to avoid overshoot
MIN_WATER_TIME = 0.5         # Minimum pump time in seconds
MAX_WATER_TIME = 10.0        # Maximum pump time in seconds

wait_time = 20  # Time to wait for moisture to stabilize after watering
is_watering = False  # Flag to prevent multiple simultaneous waterings

def send_relay_command(client, state):
    """Send relay command to turn pump on/off"""
    command = {'relay_on': state}
    print(f"Sending: {command}")
    client.publish(server_command_topic, json.dumps(command))

def calculate_water_time(current_moisture, target_moisture):
    """
    Calculate how long to run pump based on calibration data
    """
    if current_moisture <= target_moisture:
        return 0  # Already moist enough

    moisture_decrease_needed = current_moisture - target_moisture
    water_time = moisture_decrease_needed / AVG_DECREASE_PER_SEC

    # Apply safety factor to avoid overshoot
    water_time = water_time * SAFETY_FACTOR

    # Clamp to min/max values
    water_time = max(MIN_WATER_TIME, min(water_time, MAX_WATER_TIME))

    return water_time

def control_relay(client, current_moisture):
    """Control the pump based on calculated water time"""
    global is_watering

    is_watering = True

    # Calculate required water time
    water_time_needed = calculate_water_time(current_moisture, TARGET_MOISTURE)

    if water_time_needed <= 0:
        print(f"Soil moisture {current_moisture} is already below target {TARGET_MOISTURE}")
        is_watering = False
        return

    print(f"\nCurrent moisture: {current_moisture}")
    print(f"Target moisture: {TARGET_MOISTURE}")
    print(f"Decrease needed: {current_moisture - TARGET_MOISTURE}")
    print(f"Calculated pump time: {water_time_needed:.2f} seconds")

    # Unsubscribe from telemetry to ignore readings during watering
    print("Unsubscribing from telemetry...")
    mqtt_client.unsubscribe(client_telemetry_topic)

    # Run pump
    send_relay_command(client, True)
    time.sleep(water_time_needed)
    send_relay_command(client, False)

    # Wait for moisture to stabilize
    print(f"Waiting {wait_time}s for moisture to stabilize...")
    time.sleep(wait_time)

    # Re-subscribe to telemetry
    print("Resubscribing to telemetry...")
    mqtt_client.subscribe(client_telemetry_topic)

    is_watering = False
    print("Watering cycle complete\n")

def handle_telemetry(client, userdata, message):
    """Handle incoming telemetry messages"""
    global is_watering

    if is_watering:
        # Skip processing if already watering
        return

    try:
        payload = json.loads(message.payload.decode())
        print(f"📊 Message received: {payload}")

        soil_moisture = payload.get('soil_moisture')

        if soil_moisture and soil_moisture > TARGET_MOISTURE:
            print(f"Soil moisture ({soil_moisture}) exceeds target ({TARGET_MOISTURE})")
            # Start watering in a separate thread
            threading.Thread(target=control_relay, args=(client, soil_moisture), daemon=True).start()
        elif soil_moisture:
            print(f"Soil moisture ({soil_moisture}) is acceptable (target: {TARGET_MOISTURE})")

    except Exception as e:
        print(f"Error handling telemetry: {e}")

def load_calibration():
    """Load calibration data from file if it exists"""
    global AVG_DECREASE_PER_SEC
    try:
        with open('calibration_data.json', 'r') as f:
            data = json.load(f)
            AVG_DECREASE_PER_SEC = data['avg_decrease_per_sec']
            print(f"Loaded calibration: {AVG_DECREASE_PER_SEC:.2f} decrease per second")
            print(f"   Calibration date: {data.get('calibration_date', 'Unknown')}")
            return True
    except FileNotFoundError:
        print(f"No calibration file found. Using default: {AVG_DECREASE_PER_SEC}")
        print(f"   Run calibration_server.py first for accurate results")
        return False
    except Exception as e:
        print(f"Error loading calibration: {e}")
        return False

# Setup MQTT message handling
mqtt_client.subscribe(client_telemetry_topic)
mqtt_client.on_message = handle_telemetry

# Load calibration on startup
print("\n" + "="*50)
print("SOIL MOISTURE CONTROL SERVER")
print("="*50)
calibration_loaded = load_calibration()
print(f"Target moisture: {TARGET_MOISTURE}")
print(f"Safety factor: {SAFETY_FACTOR}")
print(f"Min/Max water time: {MIN_WATER_TIME}s / {MAX_WATER_TIME}s")
print("="*50 + "\n")

if not calibration_loaded:
    print("TIP: Create calibration_data.json with your calibration values")
    print("Example calibration data:")
    print(json.dumps({
        'avg_decrease_per_sec': 18.83,
        'calibration_date': '2026-04-15'
    }, indent=2))
    print()

# Main loop
try:
    while True:
        time.sleep(2)
except KeyboardInterrupt:
    print("\n\nServer stopped")
    mqtt_client.loop_stop()
    mqtt_client.disconnect()


SOIL MOISTURE CONTROL SERVER
Loaded calibration: 18.83 decrease per second
   Calibration date: 2026-04-15
Target moisture: 450
Safety factor: 0.9
Min/Max water time: 0.5s / 10.0s



Server stopped
